# Cloning the Repo

In [ ]:
!git clone https://github.com/Tahoni01/Continual-hate-speech-detection.git
%cd Continual-hate-speech-detection

# Install all the necessary library

In [2]:
%pip install -r requirements.txt

Note: you may need to restart the kernel to use updated packages.


In [3]:
import torch
from dataset.df_loader import (getdf_davidson, getdf_hatexplain)
from dataset.stream_generator import (create_continual_stream, online_stream)
from transformers import (AutoTokenizer, AutoConfig)
from models.model_builder import CustomClassifier
from models.trainer import ContinualTrainer

c:\Uni\Continual-hate-speech-detection\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Check if a GPU is avaible

In [4]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print("DEVICE:", device)

DEVICE: cpu


### Getting the datasets

In [5]:
print("Loading datasets...")

df_dv = getdf_davidson()
df_hx = getdf_hatexplain()

print("Davidson:", df_dv.shape)
print("HateXplain:", df_hx.shape)

Loading datasets...
                                                text      label    source
0  rt as a woman you shouldnt complain about clea...     normal  davidson
1  rt boy dats coldtyga dwn bad for cuffin dat ho...  offensive  davidson
2  rt dawg rt you ever fuck a bitch and she start...  offensive  davidson
3                          rt she look like a tranny  offensive  davidson
4  rt the shit you hear about me might be true or...  offensive  davidson
                                                text       label      source
0  i dont think im getting my baby them white 9 h...      normal  hatexplain
1  we cannot continue calling ourselves feminists...      normal  hatexplain
2                      nawt yall niggers ignoring me      normal  hatexplain
3  user i am bit confused coz chinese ppl can not...  hatespeech  hatexplain
4  this bitch in whataburger eating a burger with...  hatespeech  hatexplain
Davidson: (24275, 3)
HateXplain: (20144, 3)


### Creating the datastream

In [13]:
full_stream = create_continual_stream(
    df_list=[df_dv, df_hx],
    batch_size=2
)

### Creating the tokenizer and the pretrained model + custom head

In [14]:
tokenizer = AutoTokenizer.from_pretrained("distilroberta-base")

model = CustomClassifier(
    model_name="distilroberta-base",
    num_labels=3,
    )

model.to(device)

CustomClassifier(
  (backbone): RobertaModel(
    (embeddings): RobertaEmbeddings(
      (word_embeddings): Embedding(50265, 768, padding_idx=1)
      (position_embeddings): Embedding(514, 768, padding_idx=1)
      (token_type_embeddings): Embedding(1, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): RobertaEncoder(
      (layer): ModuleList(
        (0-5): 6 x RobertaLayer(
          (attention): RobertaAttention(
            (self): RobertaSdpaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): RobertaSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm):

## Training Session

In [15]:
import matplotlib.pyplot as plt
from models.trainer import ContinualTrainer
from strategy.replay import ReplayStrategy
from strategy.distillation import DistillationStrategy
from strategy.ewc import EWCStrategy
from utils.metrics import compute_accuracy, compute_f1
from utils.plot_utils import plot_loss, plot_batch_metrics, plot_conf_matrix, plot_classwise_accuracy, plot_prediction_distribution

batch_size = 2

# label map
label_map = {"hatespeech": 0, "offensive": 1, "normal": 2}

# Strategy
strategy = ReplayStrategy(buffer_size=50)  # buffer piccolo per test
#strategy = DistillationStrategy(model=model, alpha=0.5)
#strategy = EWCStrategy(model=model, lambda_=0.4)

# Trainer
trainer = ContinualTrainer(
    model=model,
    tokenizer=tokenizer,
    device=device,
)

trainer.replay = strategy if isinstance(strategy, ReplayStrategy) else None
trainer.distillation = strategy if isinstance(strategy, DistillationStrategy) else None
trainer.ewc = strategy if isinstance(strategy, EWCStrategy) else None

# Test stream
test_stream = full_stream[:10]  # 2 batch da 10 elementi ciascuno, solo per test

# Train
#losses, preds, labels = trainer.train_continual(test_stream, log_every=1)
losses, preds_list, labels_list = trainer.train_continual(
    full_stream,
    label_map=label_map
)

# Metriche globali
flat_preds = torch.stack(preds_list)
flat_labels = torch.stack(labels_list)

acc = compute_accuracy(flat_preds, flat_labels)
f1 = compute_f1(flat_preds, flat_labels)

print(f"\nAccuracy globale: {acc:.4f}, F1 globale: {f1:.4f}")

# Predizioni batch
for i, (p, l) in enumerate(zip(preds_list, labels_list)):
    print(f"\nBatch {i+1} predizioni:", p.tolist())
    print(f"Batch {i+1} etichette:", l.tolist())

plot_loss(losses)
plot_batch_metrics(flat_preds, flat_labels)
plot_conf_matrix(flat_preds, flat_labels, label_map)
plot_classwise_accuracy(flat_preds, flat_labels, label_map)
#plot_prediction_distribution(flat_preds, label_map)

  0%|          | 13/22210 [00:00<20:21, 18.17it/s]

[10] loss: 1.1595


  0%|          | 23/22210 [00:01<19:16, 19.18it/s]

[20] loss: 0.9426


  0%|          | 33/22210 [00:01<19:35, 18.86it/s]

...

[650] loss: 0.3628


  3%|▎         | 663/22210 [00:38<22:40, 15.84it/s]

[660] loss: 0.4059


  3%|▎         | 668/22210 [00:38<20:53, 17.19it/s]


KeyboardInterrupt: 